In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# CODET5 + DEVIGN : ONE-CELL FINAL CORRECTED CODE (KAGGLE)
# ============================================================

# --------------------
# Imports
# --------------------
import os
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = "/kaggle/input/devign-t5"

print("\nFiles in dataset:")
files = os.listdir(DATASET_PATH)
for f in files:
    print(" -", f)

# ============================================================
# AUTO-DETECT DEVIGN FILES
# ============================================================

def find_file(keywords):
    for f in files:
        name = f.lower()
        if all(k in name for k in keywords) and f.endswith(".csv"):
            return f
    return None

train_file = find_file(["train"])
val_file   = find_file(["val"]) or find_file(["valid"])
test_file  = find_file(["test"])

if not train_file or not test_file:
    raise FileNotFoundError("Could not detect train/test files")

print("\nDetected files:")
print("Train:", train_file)
print("Val  :", val_file)
print("Test :", test_file)

# ============================================================
# LOAD DEVIGN CSV
# ============================================================

def load_devign_csv(path):
    df = pd.read_csv(path)

    if "func" in df.columns:
        df["code"] = df["func"]
    elif "code" not in df.columns:
        raise ValueError("No code column found")

    if "target" in df.columns:
        df["label"] = df["target"]
    elif "label" not in df.columns:
        raise ValueError("No label column found")

    df = df[["code", "label"]]
    df["label"] = df["label"].astype(int)
    return df

print("\nLoading Devign dataset...")

train_df = load_devign_csv(f"{DATASET_PATH}/{train_file}")
val_df   = load_devign_csv(f"{DATASET_PATH}/{val_file}") if val_file else train_df.sample(frac=0.1, random_state=42)
test_df  = load_devign_csv(f"{DATASET_PATH}/{test_file}")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("\nTrain label distribution:\n", train_df["label"].value_counts())

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# ============================================================
# TOKENIZER (🔥 CORRECT FOR CODET5)
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

# ============================================================
# MODEL (🔥 CORRECT FOR CODET5)
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "Salesforce/codet5-base",
    num_labels=2
).to(device)

# ============================================================
# TRAINING ARGUMENTS (LOW LOSS + STABLE)
# ============================================================

training_args = TrainingArguments(
    output_dir="./codet5_devign",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    learning_rate=1e-5,               # 🔥 best for CodeT5
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    gradient_accumulation_steps=2,    # effective batch = 16
    save_strategy="no",               # Kaggle-safe
    logging_steps=100,
    report_to="none"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting CodeT5 training on Devign...")
trainer.train()

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\nEvaluating on test set...")

preds = trainer.predict(test_ds)

logits = preds.predictions
if isinstance(logits, tuple):
    logits = logits[0]

y_true = preds.label_ids
y_pred = np.argmax(logits, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== FINAL CODET5 DEVIGN RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)
print("Prediction distribution:", np.unique(y_pred, return_counts=True))

# ============================================================
# SAVE RESULTS (LOW DISK SAFE)
# ============================================================

results = {
    "dataset": "Devign",
    "model": "CodeT5",
    "epochs": 6,
    "learning_rate": 1e-5,
    "effective_batch_size": 16,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/CodeT5_Devign_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


Device: cuda
GPU: Tesla T4

Files in dataset:
 - Devignx_validation.csv
 - devignx_test.csv
 - devignx_train.csv

Detected files:
Train: devignx_train.csv
Val  : Devignx_validation.csv
Test : devignx_test.csv

Loading Devign dataset...
Train: (19122, 2)
Val  : (2732, 2)
Test : (2732, 2)

Train label distribution:
 label
0    10356
1     8766
Name: count, dtype: int64


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/19122 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

...


Starting CodeT5 training on Devign...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.778600
200,0.765600
300,0.752500
400,0.746100
500,0.732800
600,0.714900
700,0.707800
800,0.707200
900,0.704600
1000,0.703900



Evaluating on test set...



===== FINAL CODET5 DEVIGN RESULTS =====
Accuracy : 0.5728404099560761
Precision: 0.5345809601301872
Recall   : 0.5247603833865815
F1 Score : 0.5296251511487303
FPR      : 0.3864864864864865
Confusion Matrix: 908 572 595 657
Prediction distribution: (array([0, 1]), array([1503, 1229]))

Results saved to: /kaggle/working/CodeT5_Devign_metrics.json
